# 08 - Pipeline Integrado de Seguridad con IA

Este cuaderno implementa un pipeline completo de seguridad que integra todos los módulos anteriores, más la detección de deriva de datos en producción.

**Contenido:**
- Arquitectura del sistema end-to-end
- Clase `PipelineSeguridad` (detección de anomalías + clasificación de malware)
- Detección de deriva de datos con test Kolmogorov-Smirnov
- Demo completa

## 10.1 Arquitectura del sistema

```
Fuentes          Ingesta          Análisis IA          Decisión          Acción
─────────────    ────────────     ─────────────────    ──────────────    ──────────────
Logs de red  →   Pipeline ETL  →  Anomaly Detection →  Motor de reglas → Aislar sistema
Endpoints    →   Normalización →  Malware Detection →  Triaje SVM      → Bloquear IP
SIEM         →   Feature Eng.  →  UBA               →  Severidad       → Notificar CSIRT
```

## 10.2 Pipeline completo en Python

In [ ]:
import pandas as pd
import numpy as np
import joblib
import logging
import os
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)-8s %(message)s'
)


class PipelineSeguridad:
    """
    Pipeline integral de seguridad basado en IA.
    Integra detección de anomalías y clasificación de malware.
    """

    FEATURE_COLS = ['bytes_sent', 'bytes_recv', 'duration', 'src_port', 'dst_port']

    def __init__(self):
        self.scaler      = MinMaxScaler()
        self.anomaly_det = IsolationForest(
            contamination=0.05, random_state=42, n_jobs=-1
        )
        self.malware_clf = RandomForestClassifier(
            n_estimators=200, random_state=42, n_jobs=-1
        )
        self.entrenado   = False

    # ------------------------------------------------------------------
    # Entrenamiento
    # ------------------------------------------------------------------
    def entrenar(self,
                 df_trafico: pd.DataFrame,
                 df_malware: pd.DataFrame,
                 y_malware: pd.Series) -> None:
        """Entrena ambos modelos con datos históricos."""
        # Usar solo columnas disponibles
        cols_trafico = [c for c in self.FEATURE_COLS if c in df_trafico.columns]
        X_trafico = self.scaler.fit_transform(df_trafico[cols_trafico])
        self.anomaly_det.fit(X_trafico)
        logging.info('Isolation Forest entrenado.')

        self.malware_clf.fit(df_malware, y_malware)
        logging.info('Random Forest (malware) entrenado.')
        self.entrenado = True

    # ------------------------------------------------------------------
    # Persistencia
    # ------------------------------------------------------------------
    def guardar(self, ruta: str = 'pipeline_seguridad.pkl') -> None:
        joblib.dump(self, ruta)
        logging.info(f'Pipeline guardado en {ruta}')

    @staticmethod
    def cargar(ruta: str = 'pipeline_seguridad.pkl') -> 'PipelineSeguridad':
        return joblib.load(ruta)

    # ------------------------------------------------------------------
    # Inferencia
    # ------------------------------------------------------------------
    def analizar_trafico(self, df: pd.DataFrame) -> pd.DataFrame:
        """Detecta anomalías en tráfico de red."""
        if not self.entrenado:
            raise RuntimeError('El pipeline no ha sido entrenado.')
        cols = [c for c in self.FEATURE_COLS if c in df.columns]
        X    = self.scaler.transform(df[cols])
        df   = df.copy()
        df['anomalia']       = self.anomaly_det.predict(X)
        df['score_anomalia'] = self.anomaly_det.decision_function(X)
        return df

    def analizar_archivo(self, caracteristicas: dict) -> dict:
        """Clasifica un archivo como benigno o malicioso."""
        if not self.entrenado:
            raise RuntimeError('El pipeline no ha sido entrenado.')
        X    = pd.DataFrame([caracteristicas])
        pred = self.malware_clf.predict(X)[0]
        proba = self.malware_clf.predict_proba(X)[0]
        return {
            'clasificacion'  : 'malicioso' if pred == 1 else 'benigno',
            'prob_benigno'   : round(proba[0], 4),
            'prob_malicioso' : round(proba[1], 4),
        }


print('Clase PipelineSeguridad definida correctamente.')

## 10.3 Generación de datos sintéticos y entrenamiento

In [ ]:
from sklearn.preprocessing import LabelEncoder

# ---------------------------------------------------------------
# Generar datasets sintéticos si no existen
# ---------------------------------------------------------------
rng = np.random.default_rng(42)

if not os.path.exists('network_traffic.csv'):
    n = 2000
    df_net = pd.DataFrame({
        'bytes_sent' : rng.exponential(5000, n),
        'bytes_recv' : rng.exponential(8000, n),
        'duration'   : rng.exponential(30, n),
        'src_port'   : rng.integers(1024, 65535, n),
        'dst_port'   : rng.choice([80, 443, 22, 8080, 3306], n),
        'protocol'   : rng.choice(['TCP', 'UDP', 'ICMP'], n),
    })
    df_net.to_csv('network_traffic.csv', index=False)
    print('network_traffic.csv generado.')

if not os.path.exists('file_features.csv'):
    n_b, n_m = 800, 200
    benign = pd.DataFrame({
        'entry_point': rng.integers(4096, 8192, n_b),
        'num_sections': rng.integers(3, 7, n_b),
        'entropia_max': rng.uniform(4.0, 6.5, n_b),
        'num_importaciones': rng.integers(50, 200, n_b),
        'file_size': rng.integers(50000, 500000, n_b),
        'label': 0
    })
    malicious = pd.DataFrame({
        'entry_point': rng.integers(4096, 8192, n_m),
        'num_sections': rng.integers(5, 12, n_m),
        'entropia_max': rng.uniform(6.5, 8.0, n_m),
        'num_importaciones': rng.integers(200, 500, n_m),
        'file_size': rng.integers(100000, 1000000, n_m),
        'label': 1
    })
    pd.concat([benign, malicious]).to_csv('file_features.csv', index=False)
    print('file_features.csv generado.')

# Cargar y preparar datos
df_trafico = pd.read_csv('network_traffic.csv').dropna()
if 'protocol' in df_trafico.columns:
    df_trafico['protocol'] = LabelEncoder().fit_transform(df_trafico['protocol'])

df_malware = pd.read_csv('file_features.csv').dropna()
y_malware  = df_malware.pop('label')

# Instanciar y entrenar el pipeline
pipeline = PipelineSeguridad()
pipeline.entrenar(df_trafico, df_malware, y_malware)
pipeline.guardar()
print('Pipeline entrenado y guardado.')

## 10.4 Análisis en tiempo real

In [ ]:
import matplotlib.pyplot as plt

# Simular tráfico en tiempo real (muestra del dataset)
df_nuevo = df_trafico.sample(200, random_state=99).reset_index(drop=True)
resultado = pipeline.analizar_trafico(df_nuevo)

anomalias = resultado[resultado['anomalia'] == -1]
normales  = resultado[resultado['anomalia'] ==  1]

logging.info(f'Anomalías detectadas en tiempo real: {len(anomalias)}')
print(f'\nTotal analizado : {len(resultado)}')
print(f'Normales        : {len(normales)}')
print(f'Anomalías       : {len(anomalias)}')

# Visualización
plt.figure(figsize=(12, 5))
plt.scatter(range(len(normales)),
            normales['bytes_sent'], s=5, c='steelblue', alpha=0.5, label='Normal')
plt.scatter(anomalias.index,
            anomalias['bytes_sent'], s=30, c='red', alpha=0.9, label='Anomalía')
plt.xlabel('Índice')
plt.ylabel('bytes_sent')
plt.title('Análisis de tráfico en tiempo real')
plt.legend()
plt.tight_layout()
plt.savefig('pipeline_realtime.png', dpi=150)
plt.show()
print('Gráfico guardado: pipeline_realtime.png')

In [ ]:
# Análisis de un ejecutable
feats_nuevo_archivo = {
    'entry_point'       : 4096,
    'num_sections'      : 7,
    'entropia_max'      : 7.8,
    'num_importaciones' : 250,
    'file_size'         : 512000,
}
res = pipeline.analizar_archivo(feats_nuevo_archivo)
logging.info(
    f"Archivo => {res['clasificacion']} "
    f"(P(mal)={res['prob_malicioso']:.4f})"
)
print(f"\nResultado del análisis de archivo:")
for k, v in res.items():
    print(f'  {k}: {v}')

## 9.3 Detección de deriva de datos en producción

El test de **Kolmogorov-Smirnov** compara la distribución de los datos de producción con los datos de referencia para detectar cambios significativos.

In [ ]:
from scipy.stats import ks_2samp

def detectar_deriva(datos_referencia: pd.DataFrame,
                    datos_produccion: pd.DataFrame,
                    umbral_p: float = 0.05) -> dict:
    """
    Compara distribuciones de producción con los datos de referencia
    usando el test de Kolmogorov-Smirnov.
    Retorna un diccionario con columnas afectadas.
    """
    resultados = {}
    for columna in datos_referencia.columns:
        if columna not in datos_produccion.columns:
            continue
        stat, p_valor = ks_2samp(
            datos_referencia[columna].dropna(),
            datos_produccion[columna].dropna()
        )
        hay_deriva = p_valor < umbral_p
        resultados[columna] = {
            'statistic': round(stat, 4),
            'p_value'  : round(p_valor, 6),
            'deriva'   : hay_deriva
        }

    columnas_con_deriva = [
        c for c, r in resultados.items() if r['deriva']
    ]
    if columnas_con_deriva:
        print(f'[ALERTA] Deriva detectada en: {columnas_con_deriva}')
    else:
        print('[OK] Sin deriva significativa en los datos de entrada.')
    return resultados


# Simular datos de referencia y producción
feature_cols = ['bytes_sent', 'bytes_recv', 'duration', 'src_port']
feature_cols = [c for c in feature_cols if c in df_trafico.columns]

df_ref  = df_trafico[feature_cols].iloc[:1000]

# Producción normal (sin deriva)
df_prod_normal = df_trafico[feature_cols].iloc[1000:1500]
print('=== Producción normal ===')
resultado_normal = detectar_deriva(df_ref, df_prod_normal)

# Producción con deriva (multiplicar bytes_sent por 10)
df_prod_deriva = df_trafico[feature_cols].iloc[1000:1500].copy()
if 'bytes_sent' in df_prod_deriva.columns:
    df_prod_deriva['bytes_sent'] *= 10
print('\n=== Producción con deriva en bytes_sent ===')
resultado_deriva = detectar_deriva(df_ref, df_prod_deriva)

# Mostrar tabla de resultados
print('\n=== Detalle de resultados (producción con deriva) ===')
df_resultado = pd.DataFrame(resultado_deriva).T
print(df_resultado.to_string())

## 10.5 Cargar pipeline guardado y reutilizar

In [ ]:
# Cargar pipeline desde disco
pipeline_cargado = PipelineSeguridad.cargar('pipeline_seguridad.pkl')
logging.info('Pipeline cargado desde disco.')

# Verificar que funciona
res_verificacion = pipeline_cargado.analizar_archivo(feats_nuevo_archivo)
print(f"Pipeline cargado correctamente. Clasificación: {res_verificacion['clasificacion']}")

## Resumen del pipeline integrado

| Módulo | Técnica | Salida |
|---|---|---|
| Detección de anomalías | Isolation Forest | anomalia (-1/1) + score |
| Clasificación de malware | Random Forest | benigno/malicioso + probabilidades |
| Deriva de datos | Test KS | columnas con deriva |
| Persistencia | joblib | pipeline_seguridad.pkl |

**Próximos pasos recomendados:**
1. Conectar con SIEM/SOAR real (ver notebook 04)
2. Añadir capa de explicabilidad SHAP (ver notebook 06)
3. Evaluar robustez adversarial (ver notebook 07)
4. Configurar reentrenamiento periódico con nuevas muestras